# Week 5 – Milestone 1: Smart Logistics Tracking — Blockchain Ledger
**MO-IT148 | Smart Tracking System**

---
## SECTION 1: SETTING UP THE ENVIRONMENT

In [1]:
from web3 import Web3

# Connect to local Ganache blockchain
ganache_url = "http://127.0.0.1:7545"
web3 = Web3(Web3.HTTPProvider(ganache_url))

print("========================================")
print("  SETTING UP THE ENVIRONMENT")
print("========================================")

if web3.is_connected():
    print("✅ Connected to Ganache successfully!")
    print(f"   Was your connection successful? YES")
    print(f"   Chain ID  : {web3.eth.chain_id}")
    print(f"   Block #   : {web3.eth.block_number}")
    print(f"   Accounts  : {len(web3.eth.accounts)} available")
else:
    print("❌ Connection failed. Ensure Ganache is running.")
    print("   Was your connection successful? NO")

  SETTING UP THE ENVIRONMENT
✅ Connected to Ganache successfully!
   Was your connection successful? YES
   Chain ID  : 1337
   Block #   : 1
   Accounts  : 10 available


---
## SECTION 2: SMART CONTRACT INTERACTION

In [2]:
# ── REPLACE CONTRACT ADDRESS WITH YOUR OWN FROM REMIX ──────────────
contract_address = "0xbAa81700cA3544e0c14C97F3BB35C8A49d9dB14a"

abi = [
    {
        "inputs": [{"internalType": "uint256", "name": "index", "type": "uint256"}],
        "name": "getRecord",
        "outputs": [
            {"internalType": "string", "name": "", "type": "string"},
            {"internalType": "string", "name": "", "type": "string"},
            {"internalType": "string", "name": "", "type": "string"}
        ],
        "stateMutability": "view",
        "type": "function"
    },
    {
        "inputs": [],
        "name": "getTotalRecords",
        "outputs": [{"internalType": "uint256", "name": "", "type": "uint256"}],
        "stateMutability": "view",
        "type": "function"
    },
    {
        "inputs": [
            {"internalType": "string", "name": "_deviceId", "type": "string"},
            {"internalType": "string", "name": "_dataType",  "type": "string"},
            {"internalType": "string", "name": "_value",     "type": "string"}
        ],
        "name": "storeData",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function"
    }
]
# ───────────────────────────────────────────────────────────────────

contract_address = Web3.to_checksum_address(contract_address)
contract = web3.eth.contract(address=contract_address, abi=abi)
web3.eth.default_account = web3.eth.accounts[0]

total_before = contract.functions.getTotalRecords().call()

print("========================================")
print("  SMART CONTRACT INTERACTION")
print("========================================")
print(f"✅ Smart contract connected!")
print(f"   Smart Contract Address          : {contract_address}")
print(f"   Default Sender Account          : {web3.eth.default_account}")
print(f"   Total Records Before Storing    : {total_before}")
print(f"   Errors while loading contract?  : NO")

  SMART CONTRACT INTERACTION
✅ Smart contract connected!
   Smart Contract Address          : 0xbAa81700cA3544e0c14C97F3BB35C8A49d9dB14a
   Default Sender Account          : 0x923f4EB4205324f33e1750E9FC9e7eD61206c5E8
   Total Records Before Storing    : 0
   Errors while loading contract?  : NO


---
## SECTION 3: STORING IoT DATA ON THE BLOCKCHAIN
### Part 1 – Load CSV File (with Sensor Types)

In [3]:
import pandas as pd

# Load the updated IoT dataset with sensor_type column
df = pd.read_csv("iot_data_final.csv")

print("========================================")
print("  STORING IoT DATA — CSV PREVIEW")
print("========================================")
print(f"   Number of records in CSV file : {len(df)}")
print(f"   Sensor types available        : {df['sensor_type'].unique().tolist()}")
print(f"   Records per sensor type       : {df['sensor_type'].value_counts().to_dict()}")
print()
print("   First 3 records in CSV:")
print(f"   {'#':<5} {'RFID Tag':<10} {'Sensor Type':<14} {'Value':<12} {'Package':<10} {'Status'}")
print("   " + "-" * 65)
for i, row in df.head(3).iterrows():
    print(f"   {i+1:<5} {row['rfid_tag']:<10} {row['sensor_type']:<14} {row['data_value']:<12} PKG{row['package_id']:<6} {row['status']}")

  STORING IoT DATA — CSV PREVIEW
   Number of records in CSV file : 300
   Sensor types available        : ['Temperature', 'Humidity', 'GPS']
   Records per sensor type       : {'Temperature': 100, 'Humidity': 100, 'GPS': 100}

   First 3 records in CSV:
   #     RFID Tag   Sensor Type    Value        Package    Status
   -----------------------------------------------------------------
   1     RFID_0     Temperature    3.82°C       PKG1064   In Transit
   2     RFID_0     Humidity       50.97%       PKG1064   In Transit
   3     RFID_0     GPS            12.422315329212937,121.19571255545216 PKG1064   In Transit


### Part 2 – Store Each Row as a Blockchain Transaction

In [4]:
import time

def send_iot_data(device_id, sensor_type, data_value):
    """Sends IoT sensor data to the deployed smart contract.
    
    Args:
        device_id   : RFID tag (e.g. RFID_0)
        sensor_type : Temperature | Humidity | GPS
        data_value  : sensor reading (e.g. 3.82°C, 50.97%, lat,lon)
    """
    txn = contract.functions.storeData(
        device_id, sensor_type, data_value
    ).transact({
        'from': web3.eth.default_account,
        'gas': 3000000
    })
    receipt = web3.eth.wait_for_transaction_receipt(txn)
    print(f"✅ [{sensor_type:<12}] {device_id} | {data_value}")
    print(f"   Txn Hash: {receipt.transactionHash.hex()}")
    return receipt

print("========================================")
print("  STORING IoT DATA — TRANSACTIONS")
print("========================================")
print(f"Sending {len(df)} records to blockchain...")
print(f"(100 Temperature + 100 Humidity + 100 GPS)")
print()

first_txn_hash = None
last_txn_hash  = None
success_count  = 0

for i, (_, row) in enumerate(df.iterrows()):
    receipt = send_iot_data(
        str(row['rfid_tag']),
        str(row['sensor_type']),
        str(row['data_value'])
    )
    if i == 0:
        first_txn_hash = receipt.transactionHash.hex()
    last_txn_hash = receipt.transactionHash.hex()
    success_count += 1
    time.sleep(0.5)

print()
print("========================================")
print("  TRANSACTION SUMMARY")
print("========================================")
print(f"   Records successfully stored         : {success_count}")
print(f"   Transaction Hash of FIRST record    : {first_txn_hash}")
print(f"   Transaction Hash of LAST  record    : {last_txn_hash}")

  STORING IoT DATA — TRANSACTIONS
Sending 300 records to blockchain...
(100 Temperature + 100 Humidity + 100 GPS)

✅ [Temperature ] RFID_0 | 3.82°C
   Txn Hash: 381be323a9fa430fb539087c17fd02a5a66a5a40c7dafecd65dabb247487c049
✅ [Humidity    ] RFID_0 | 50.97%
   Txn Hash: 894144db6431a5c4c443e4c196e97d916f58f31f638a1d4329d12c891d4f9078
✅ [GPS         ] RFID_0 | 12.422315329212937,121.19571255545216
   Txn Hash: dc9b696959721a9b43e50c7b7e462158d2d94e78169fdc6a0d45dc95e54aa4ab
✅ [Temperature ] RFID_1 | 2.93°C
   Txn Hash: 7950d76df7c1e5cfcd6f3099fc1f2b9690833e6b7927a886132d7b94a2d2cc71
✅ [Humidity    ] RFID_1 | 60.65%
   Txn Hash: 9b4df6dd08093035e2f1c823bc9d8e93235ce45d18c3960f15453e41d37010f3
✅ [GPS         ] RFID_1 | 13.513377156951517,122.51412936901205
   Txn Hash: 2ee9e349e5c35b3966a48737c41b8670270d95579996ffb6b5eae99076256126
✅ [Temperature ] RFID_2 | 5.47°C
   Txn Hash: 32f9ebe424b82a7b837db6abffb4c4dfd88c44df53016fe61f04fb3cc0d30fd8
✅ [Humidity    ] RFID_2 | 88.83%
   Txn Hash: 

---
## SECTION 4: RETRIEVING & VERIFYING DATA

In [5]:
import datetime

total_after = contract.functions.getTotalRecords().call()
first_record = contract.functions.getRecord(0).call()

latest_block = web3.eth.get_block('latest')
timestamp = datetime.datetime.fromtimestamp(
    latest_block['timestamp'], datetime.timezone.utc
).strftime("%Y-%m-%d %H:%M:%S")

print("========================================")
print("  RETRIEVING & VERIFYING DATA")
print("========================================")
print(f"   Total records now stored on blockchain : {total_after}")
print()
print("   First stored record retrieved from blockchain:")
print(f"   Timestamp   : {timestamp} UTC")
print(f"   Device ID   : {first_record[0]}")
print(f"   Sensor Type : {first_record[1]}")
print(f"   Data Value  : {first_record[2]}")

  RETRIEVING & VERIFYING DATA
   Total records now stored on blockchain : 300

   First stored record retrieved from blockchain:
   Timestamp   : 2026-07-04 02:18:29 UTC
   Device ID   : RFID_0
   Sensor Type : Temperature
   Data Value  : 3.82°C


---
## SECTION 5: SENSOR TYPE FILTER — ALL RECORDS TABLE

In [6]:
# Retrieve all records from blockchain
total = contract.functions.getTotalRecords().call()
all_records = []

print(f"Retrieving all {total} records from blockchain...")
for i in range(total):
    r = contract.functions.getRecord(i).call()
    all_records.append({
        "Index":       i,
        "Device ID":   r[0],
        "Sensor Type": r[1],
        "Data Value":  r[2]
    })

result_df = pd.DataFrame(all_records)

print(f"\n✅ All {total} records retrieved!")
print(f"   Records per sensor type:")
print(result_df['Sensor Type'].value_counts().to_string())
print()
result_df

Retrieving all 300 records from blockchain...

✅ All 300 records retrieved!
   Records per sensor type:
Sensor Type
Temperature    100
Humidity       100
GPS            100



,Index,Device ID,Sensor Type,Data Value
0,0,RFID_0,Temperature,3.82°C
1,1,RFID_0,Humidity,50.97%
2,2,RFID_0,GPS,"12.422315329212937,121.19571255545216"
3,3,RFID_1,Temperature,2.93°C
4,4,RFID_1,Humidity,60.65%
...,...,...,...,...
295,295,RFID_98,Humidity,84.53%
296,296,RFID_98,GPS,"12.31157451159378,124.78112434774869"
297,297,RFID_99,Temperature,6.81°C
298,298,RFID_99,Humidity,50.13%


In [7]:
# ── SENSOR TYPE FILTER ──────────────────────────────────────────────
# Change this to: "Temperature", "Humidity", or "GPS"
FILTER_SENSOR = "Temperature"
# ───────────────────────────────────────────────────────────────────

filtered = result_df[result_df['Sensor Type'] == FILTER_SENSOR]

print(f"========================================")
print(f"  SENSOR TYPE FILTER: {FILTER_SENSOR}")
print(f"========================================")
print(f"   Records found : {len(filtered)}")
print()
filtered

  SENSOR TYPE FILTER: Temperature
   Records found : 100



,Index,Device ID,Sensor Type,Data Value
0,0,RFID_0,Temperature,3.82°C
3,3,RFID_1,Temperature,2.93°C
6,6,RFID_2,Temperature,5.47°C
9,9,RFID_3,Temperature,4.44°C
12,12,RFID_4,Temperature,2.58°C
...,...,...,...,...
285,285,RFID_95,Temperature,7.74°C
288,288,RFID_96,Temperature,7.37°C
291,291,RFID_97,Temperature,3.93°C
294,294,RFID_98,Temperature,6.36°C
